# Models Pipeline

This notebook is the restartable entry point for model estimation, pooled out-of-sample prediction, diagnostic evaluation, portfolio construction, implementability analysis, and paired model tests. Annual refits and derived artifacts are cached by model ID and specification signature.

## Model roster

The model roster includes conventional linear, tree-based, and neural benchmarks; the LightGBM 20/40/60/80/100 characteristic expansion; exact-calendar lagged LightGBM specifications; independent-stock MLP controls; permutation-invariant DeepSets models; and validation-weighted hybrids. `NN2_20` uses hidden widths `[32, 16]`, `NN3_20` uses `[32, 16, 8]`, and `NN4_20` uses `[32, 16, 8, 4]`. These are GKX-style benchmarks rather than exact replications because the information set and sample differ. `MLP_40` is the monthly-panel control without pooled market context. `DEEPSET_40_LAG1` adds exact one-month-lagged characteristics, and `DEEPSET_40_DYNAMIC` additionally includes one-month characteristic velocities.

## Fixed design decisions

- The target is decimal next-month excess return, `ret_exc_lead1m`.
- The universe is USA stocks with a valid stable JKP `id` and size group micro/small/large/mega; nano stocks are excluded.
- Characteristics are ranked within the complete eligible monthly cross-section and mapped to `[-1, 1]` before target availability is inspected. Missing targets are masked from fitting and evaluation calculations but do not change the month-t ranking universe.
- Exact-calendar lags are joined by JKP `id`. Missing prior months receive neutral values plus a zero availability flag.
- The rolling design uses 15 training years, 4 validation years, one temporally held-out test year, and annual refits from 1999 through 2024.
- Validation MSE selects hyperparameters and early stopping. Test outcomes never affect fitting or model selection.
- Primary evaluation is pooled GKX OOS R-squared and the equal-weighted D10-D1 portfolio. Rank IC, calibration, robust R-squared, monotonicity, alternative portfolios, transaction costs, universe sensitivity, and an adverse missing-return stress are supporting diagnostics.
- Constant-forecast months hold cash. Prediction ties receive fractional boundary allocation, so portfolio returns do not depend on identifier ordering.
- Each model/refit has an independent signature and completion marker. Compatible results load without retraining or overwriting.

## 1. Runtime and project setup

This cell selects the Google Drive project when executed in Colab and the local project directory under a standard Windows kernel. It clears cached `src` modules and verifies the resolved import path before configuration or training.

In [1]:
import os
import sys
import importlib
from pathlib import Path

LOCAL_PROJECT_DIR = Path(r"C:\Users\sandh\OneDrive\Documents\Coding\FDS Project")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/FDS Project")

try:
    from google.colab import drive
except ImportError:
    RUNNING_IN_COLAB = False
    PROJECT_DIR = LOCAL_PROJECT_DIR
    RUNTIME = "Local VS Code kernel"
else:
    RUNNING_IN_COLAB = True
    # Reuse an existing Drive mount when available.
    drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = DRIVE_PROJECT_DIR
    RUNTIME = "Google Colab kernel in VS Code"

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"Project folder was not found: {PROJECT_DIR}\n"
        "If this is Colab, confirm that Drive is mounted and that the folder name matches exactly."
    )
if not (PROJECT_DIR / "src").is_dir():
    raise FileNotFoundError(f"The project src folder was not found under: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
project_path = str(PROJECT_DIR)
# Prioritize the selected project directory during module resolution.
sys.path = [path for path in sys.path if path != project_path]
sys.path.insert(0, project_path)
for module_name in tuple(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
importlib.invalidate_caches()

# Verify the resolved source package before configuration.
import src
src_file = Path(src.__file__).resolve()
if PROJECT_DIR.resolve() not in src_file.parents:
    raise RuntimeError(f"Imported src from the wrong location: {src_file}")

print("Notebook file: local Models_Pipeline.ipynb")
print("Runtime:", RUNTIME)
print("Project files and outputs:", PROJECT_DIR)
print("Imported src from:", src_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook file: local Models_Pipeline.ipynb
Runtime: Google Colab kernel in VS Code
Project files and outputs: /content/drive/MyDrive/Colab Notebooks/FDS Project
Imported src from: /content/drive/MyDrive/Colab Notebooks/FDS Project/src/__init__.py


## 2. Experiment configuration

The configuration fixes the dataset, experiment identifier, selected model roster, random seed, and device policy. A new experiment identifier is required after changing the universe, target, preprocessing, feature definitions, or rolling schedule.

In [2]:
if 'PROJECT_DIR' not in globals():
    raise RuntimeError('Run the Runtime and project setup cell first, or use Run all.')

from src.config import ExperimentConfig, UniverseConfig
from src.models import MODEL_REGISTRY

DATA_PATH = PROJECT_DIR / 'jkp_USA_100chars_1980_2024.parquet'
OUTPUT_DIR = PROJECT_DIR / 'model_runs'
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Raw data file not found: {DATA_PATH}')

MODEL_ROSTER = (
    'LASSO_20', 'LGBM_20', 'XGBOOST_20', 'NN2_20', 'NN3_20', 'NN4_20',
    'LGBM_40', 'LGBM_60', 'LGBM_80', 'LGBM_100',
    'LGBM_40_LAG1', 'LGBM_40_LAG2',
    'MLP_40', 'DEEPSET_40', 'DEEPSET_40_LAG1', 'DEEPSET_40_DYNAMIC',
    'HYBRID_MLP40_DEEPSET40',
    'HYBRID_LGBM40_DEEPSET40', 'HYBRID_LGBM40_DEEPSET40_DYNAMIC',
)
# Hybrid components use years 1--3 of validation for early stopping;
# the convex combination is estimated on validation year 4.
SELECTED_MODELS = MODEL_ROSTER

CONFIG = ExperimentConfig(
    experiment_id='core20_benchmarks_v1',
    project_dir=PROJECT_DIR,
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    selected_models=SELECTED_MODELS,
    seed=42,
    use_gpu=True,
    universe=UniverseConfig(security_id_col='id'),
)
CONFIG.validate()

import torch
if CONFIG.use_gpu and not torch.cuda.is_available():
    raise RuntimeError('GPU requested but unavailable. Select a GPU runtime and reconnect before training.')
print('Torch device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Selected models:', list(CONFIG.selected_models))
print('Run directory:', CONFIG.run_dir)

Torch device: NVIDIA A100-SXM4-40GB
Selected models: ['LASSO_20', 'LGBM_20', 'XGBOOST_20', 'NN2_20', 'NN3_20', 'NN4_20', 'LGBM_40', 'LGBM_60', 'LGBM_80', 'LGBM_100', 'LGBM_40_LAG1', 'LGBM_40_LAG2', 'MLP_40', 'DEEPSET_40', 'DEEPSET_40_LAG1', 'DEEPSET_40_DYNAMIC', 'HYBRID_MLP40_DEEPSET40', 'HYBRID_LGBM40_DEEPSET40', 'HYBRID_LGBM40_DEEPSET40_DYNAMIC']
Run directory: /content/drive/MyDrive/Colab Notebooks/FDS Project/model_runs/core20_benchmarks_v1


## 3. Preflight checks

This verifies required project files, registry consistency, feature construction, rolling-window timing, evaluation definitions, hybrid validation separation, and tie-safe portfolio construction before model estimation.

In [3]:
import unittest
from src.self_checks import run_framework_self_checks

required_files = (
    DATA_PATH,
    PROJECT_DIR / 'src' / 'config.py',
    PROJECT_DIR / 'src' / 'models.py',
    PROJECT_DIR / 'src' / 'runner.py',
    PROJECT_DIR / 'src' / 'self_checks.py',
    PROJECT_DIR / 'tests' / 'test_pipeline.py',
)
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f'Required project files are missing: {missing_files}')
unknown_models = sorted(set(CONFIG.selected_models) - set(MODEL_REGISTRY))
if unknown_models:
    raise ValueError(f'Selected models are not registered: {unknown_models}')

run_framework_self_checks()
suite = unittest.defaultTestLoader.discover(str(PROJECT_DIR / 'tests'))
test_result = unittest.TextTestRunner(verbosity=1).run(suite)
if not test_result.wasSuccessful():
    raise RuntimeError('Pipeline unit tests failed.')
print('Project files: PASS')
print('Selected model registry: PASS')
print('Framework self-checks: PASS')
print('Pipeline unit tests: PASS')

......................
----------------------------------------------------------------------
Ran 22 tests in 3.067s

OK


Project files: PASS
Selected model registry: PASS
Framework self-checks: PASS
Pipeline unit tests: PASS


## 4. Run or resume

This is safe to rerun. Compatible completed refits load, incomplete work resumes or reruns, pooled files rebuild from refit outputs, and metrics and portfolios refresh.

In [4]:
# Process models separately to bound panel memory use.
from dataclasses import replace
import gc
import torch
from src.runner import ExperimentRunner

for model_id in CONFIG.selected_models:
    print(f'\n=== {model_id} ===')
    model_config = replace(CONFIG, selected_models=(model_id,))
    ExperimentRunner(model_config).run()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Construct the cumulative comparison over the full roster.
runner = ExperimentRunner(CONFIG)
comparison = runner._cumulative_comparison()
display(comparison)


=== LASSO_20 ===
Device: cuda
LASSO_20 [bc11f57268346015]: current diagnostics; skipping

=== LGBM_20 ===
Device: cuda
LGBM_20 [2384b144be38bad7]: current diagnostics; skipping

=== XGBOOST_20 ===
Device: cuda
XGBOOST_20 [507f3c420a300c4b]: current diagnostics; skipping

=== NN2_20 ===
Device: cuda
NN2_20 [6130644ff7a7bc93]: current diagnostics; skipping

=== NN3_20 ===
Device: cuda
NN3_20 [739c8c490c1c6d16]: current diagnostics; skipping

=== NN4_20 ===
Device: cuda
NN4_20 [eee1133001253343]: current diagnostics; skipping

=== LGBM_40 ===
Device: cuda
LGBM_40 [9c0540e864a705bf]: current diagnostics; skipping

=== LGBM_60 ===
Device: cuda
LGBM_60 [94e3ec047ddc57fc]: current diagnostics; skipping

=== LGBM_80 ===
Device: cuda
LGBM_80 [80077a60ed146994]: current diagnostics; skipping

=== LGBM_100 ===
Device: cuda
LGBM_100 [56fb90cc59a0ce74]: current diagnostics; skipping

=== LGBM_40_LAG1 ===
Device: cuda
LGBM_40_LAG1 [4aafd74344026b96]: current diagnostics; skipping

=== LGBM_40_LAG2 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    epoch 001/100 | train_mse=0.02262973 | val_mse=0.02423941 | best_val_mse=0.02423941 | early_stop=00/10 | 1.4s
    epoch 002/100 | train_mse=0.02244136 | val_mse=0.02411871 | best_val_mse=0.02411871 | early_stop=00/10 | 1.4s
    epoch 003/100 | train_mse=0.02237749 | val_mse=0.02403725 | best_val_mse=0.02403725 | early_stop=00/10 | 1.4s
    epoch 004/100 | train_mse=0.02236861 | val_mse=0.02414176 | best_val_mse=0.02403725 | early_stop=01/10 | 1.4s
    epoch 005/100 | train_mse=0.02234328 | val_mse=0.02405427 | best_val_mse=0.02403725 | early_stop=02/10 | 1.4s
    epoch 006/100 | train_mse=0.02229313 | val_mse=0.02393028 | best_val_mse=0.02393028 | early_stop=00/10 | 1.4s
    epoch 007/100 | train_mse=0.02231871 | val_mse=0.02392097 | best_val_mse=0.02392097 | early_stop=00/10 | 1.4s
    epoch 008/100 | train_mse=0.02231310 | val_mse=0.02389111 | best_val_mse=0.02389111 | early_stop=00/10 | 1.4s
    epoch 009/100 | train_mse=0.02230357 | val_mse=0.02403970 | best_val_mse=0.02389111 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    epoch 001/100 | train_mse=0.02251642 | val_mse=0.03017240 | best_val_mse=0.03017240 | early_stop=00/10 | 1.4s
    epoch 002/100 | train_mse=0.02238712 | val_mse=0.03014401 | best_val_mse=0.03014401 | early_stop=00/10 | 1.5s
    epoch 003/100 | train_mse=0.02231027 | val_mse=0.03009150 | best_val_mse=0.03009150 | early_stop=00/10 | 1.4s
    epoch 004/100 | train_mse=0.02225045 | val_mse=0.03014038 | best_val_mse=0.03009150 | early_stop=01/10 | 1.5s
    epoch 005/100 | train_mse=0.02230536 | val_mse=0.03011235 | best_val_mse=0.03009150 | early_stop=02/10 | 1.5s
    epoch 006/100 | train_mse=0.02221583 | val_mse=0.03006889 | best_val_mse=0.03006889 | early_stop=00/10 | 1.5s
    epoch 007/100 | train_mse=0.02221761 | val_mse=0.03002922 | best_val_mse=0.03002922 | early_stop=00/10 | 1.4s
    epoch 008/100 | train_mse=0.02219261 | val_mse=0.03004135 | best_val_mse=0.03002922 | early_stop=01/10 | 1.4s
    epoch 009/100 | train_mse=0.02220608 | val_mse=0.03005928 | best_val_mse=0.03002922 

...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    epoch 001/100 | train_mse=0.02228257 | val_mse=0.04120696 | best_val_mse=0.04120696 | early_stop=00/10 | 1.4s
    epoch 002/100 | train_mse=0.02161886 | val_mse=0.04122010 | best_val_mse=0.04120696 | early_stop=01/10 | 1.4s
    epoch 003/100 | train_mse=0.02144859 | val_mse=0.04114263 | best_val_mse=0.04114263 | early_stop=00/10 | 1.4s
    epoch 004/100 | train_mse=0.02150765 | val_mse=0.04136835 | best_val_mse=0.04114263 | early_stop=01/10 | 1.4s
    epoch 005/100 | train_mse=0.02149612 | val_mse=0.04113540 | best_val_mse=0.04113540 | early_stop=00/10 | 1.4s
    epoch 006/100 | train_mse=0.02135022 | val_mse=0.04136683 | best_val_mse=0.04113540 | early_stop=01/10 | 1.4s
    epoch 007/100 | train_mse=0.02134366 | val_mse=0.04110024 | best_val_mse=0.04110024 | early_stop=00/10 | 1.4s
    epoch 008/100 | train_mse=0.02133109 | val_mse=0.04116163 | best_val_mse=0.04110024 | early_stop=01/10 | 1.4s
    epoch 009/100 | train_mse=0.02130336 | val_mse=0.04114325 | best_val_mse=0.04110024 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    epoch 001/100 | train_mse=0.02268921 | val_mse=0.04520349 | best_val_mse=0.04520349 | early_stop=00/10 | 1.4s
    epoch 002/100 | train_mse=0.02238969 | val_mse=0.04516050 | best_val_mse=0.04516050 | early_stop=00/10 | 1.4s
    epoch 003/100 | train_mse=0.02234955 | val_mse=0.04500828 | best_val_mse=0.04500828 | early_stop=00/10 | 1.4s
    epoch 004/100 | train_mse=0.02217075 | val_mse=0.04496568 | best_val_mse=0.04496568 | early_stop=00/10 | 1.4s
    epoch 005/100 | train_mse=0.02216924 | val_mse=0.04500365 | best_val_mse=0.04496568 | early_stop=01/10 | 1.4s
    epoch 006/100 | train_mse=0.02213612 | val_mse=0.04496039 | best_val_mse=0.04496039 | early_stop=00/10 | 1.4s
    epoch 007/100 | train_mse=0.02207522 | val_mse=0.04497667 | best_val_mse=0.04496039 | early_stop=01/10 | 1.4s
    epoch 008/100 | train_mse=0.02205943 | val_mse=0.04515593 | best_val_mse=0.04496039 | early_stop=02/10 | 1.4s
    epoch 009/100 | train_mse=0.02211640 | val_mse=0.04497149 | best_val_mse=0.04496039 

,model_id,model_signature,pooled_oos_r2,n_predictions,diagnostics_version,robust_oos_r2,mean_monthly_rank_ic,stock_count_weighted_mean_rank_ic,rank_ic_std,rank_ic_information_ratio,...,tail_5pct_mean_short_coverage,mean_monthly_return,annualized_return,annualized_volatility,sharpe,t_stat,newey_west_t_stat,max_drawdown,hit_rate,n_months
8,LGBM_20,2384b144be38bad7,0.003086,1356639,post_model_v6_fractional_ties,0.003808,0.060782,0.065054,0.098706,2.133144,...,0.983126,0.023449,0.281386,0.190347,1.478275,7.537755,5.577166,-0.373339,0.695513,312
18,XGBOOST_20,507f3c420a300c4b,0.002922,1356639,post_model_v6_fractional_ties,0.003854,0.062256,0.066416,0.098213,2.195861,...,0.984952,0.023188,0.278261,0.197928,1.405871,7.168563,5.435424,-0.432644,0.708333,312
9,LGBM_40,9c0540e864a705bf,0.002835,1356639,post_model_v6_fractional_ties,0.003282,0.053859,0.058450,0.089344,2.088255,...,0.978927,0.024516,0.294188,0.176287,1.668800,8.509243,5.742294,-0.318891,0.746795,312
12,LGBM_60,94e3ec047ddc57fc,0.002619,1356639,post_model_v6_fractional_ties,0.002808,0.034065,0.038211,0.110666,1.066310,...,0.976185,0.019709,0.236512,0.185318,1.276250,6.507623,4.146292,-0.361825,0.641026,312
11,LGBM_40_LAG2,5ea44d3ad39373d9,0.002538,1356639,post_model_v6_fractional_ties,0.002916,0.056310,0.061383,0.095859,2.034899,...,0.982039,0.025517,0.306208,0.184819,1.656799,8.448048,5.819175,-0.367963,0.730769,312
10,LGBM_40_LAG1,4aafd74344026b96,0.002302,1356639,post_model_v6_fractional_ties,0.002672,0.055924,0.060804,0.094822,2.043043,...,0.979866,0.025624,0.307483,0.182669,1.683278,8.583070,5.917585,-0.330912,0.753205,312
14,MLP_40,e6a25033ee712d15,0.002287,1356639,post_model_v6_fractional_ties,0.002552,0.060230,0.064485,0.089489,2.331493,...,0.990752,0.025333,0.304001,0.177616,1.711557,8.727261,5.805627,-0.289474,0.721154,312
16,NN3_20,739c8c490c1c6d16,0.001806,1356639,post_model_v6_fractional_ties,0.002120,0.051122,0.054852,0.090022,1.967208,...,0.985011,0.019546,0.234549,0.180191,1.301669,6.637238,4.855011,-0.413451,0.692308,312
6,LASSO_20,bc11f57268346015,0.001746,1356639,post_model_v6_fractional_ties,0.002109,0.071919,0.076510,0.133655,1.864029,...,0.987852,0.014327,0.171923,0.240600,0.714560,3.643554,2.969408,-0.567106,0.467949,312
5,HYBRID_MLP40_DEEPSET40,e76569f5ea5b351a,0.001529,1356639,post_model_v6_fractional_ties,0.001416,0.057928,0.061570,0.083942,2.390546,...,0.992155,0.024872,0.298459,0.170882,1.746579,8.905839,5.839902,-0.354444,0.733974,312


## 5. HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50

After `LGBM_40` and `DEEPSET_40_DYNAMIC` are complete, this forms a transparent US comparator by averaging their independently trained, observation-aligned OOS predictions. It performs no additional fitting and does not use test outcomes to choose weights.

In [5]:
import pandas as pd
from src.developed_markets import build_fifty_fifty_comparator

us_fifty_fifty = build_fifty_fifty_comparator(CONFIG)
display(pd.DataFrame([us_fifty_fifty]))

,model_id,model_signature,comparator_version,pooled_oos_r2,robust_oos_r2,mean_monthly_rank_ic,stock_count_weighted_mean_rank_ic,rank_ic_std,rank_ic_information_ratio,rank_ic_t_stat,...,rank_ic_autocorr_lag12,mean_monthly_return,annualized_return,annualized_volatility,sharpe,t_stat,newey_west_t_stat,max_drawdown,hit_rate,n_months
0,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,d14e8c2a63ec5c8e,standalone_oos_prediction_average_v1,0.003475,0.00403,0.060747,0.065044,0.077398,2.718839,13.863413,...,-0.041883,0.028445,0.34134,0.168209,2.029261,10.347244,6.382897,-0.35137,0.766026,312


## 6. Reload saved comparison 

In [6]:
import pandas as pd

comparison_path = CONFIG.run_dir / 'model_comparison.csv'
if comparison_path.exists():
    display(pd.read_csv(comparison_path))
else:
    print('No completed model comparison exists yet.')

,model_id,model_signature,pooled_oos_r2,n_predictions,diagnostics_version,robust_oos_r2,mean_monthly_rank_ic,stock_count_weighted_mean_rank_ic,rank_ic_std,rank_ic_information_ratio,...,tail_5pct_mean_short_coverage,mean_monthly_return,annualized_return,annualized_volatility,sharpe,t_stat,newey_west_t_stat,max_drawdown,hit_rate,n_months
0,LGBM_20,2384b144be38bad7,0.003086,1356639,post_model_v6_fractional_ties,0.003808,0.060782,0.065054,0.098706,2.133144,...,0.983126,0.023449,0.281386,0.190347,1.478275,7.537755,5.577166,-0.373339,0.695513,312
1,XGBOOST_20,507f3c420a300c4b,0.002922,1356639,post_model_v6_fractional_ties,0.003854,0.062256,0.066416,0.098213,2.195861,...,0.984952,0.023188,0.278261,0.197928,1.405871,7.168563,5.435424,-0.432644,0.708333,312
2,LGBM_40,9c0540e864a705bf,0.002835,1356639,post_model_v6_fractional_ties,0.003282,0.053859,0.058450,0.089344,2.088255,...,0.978927,0.024516,0.294188,0.176287,1.668800,8.509243,5.742294,-0.318891,0.746795,312
3,LGBM_60,94e3ec047ddc57fc,0.002619,1356639,post_model_v6_fractional_ties,0.002808,0.034065,0.038211,0.110666,1.066310,...,0.976185,0.019709,0.236512,0.185318,1.276250,6.507623,4.146292,-0.361825,0.641026,312
4,LGBM_40_LAG2,5ea44d3ad39373d9,0.002538,1356639,post_model_v6_fractional_ties,0.002916,0.056310,0.061383,0.095859,2.034899,...,0.982039,0.025517,0.306208,0.184819,1.656799,8.448048,5.819175,-0.367963,0.730769,312
5,LGBM_40_LAG1,4aafd74344026b96,0.002302,1356639,post_model_v6_fractional_ties,0.002672,0.055924,0.060804,0.094822,2.043043,...,0.979866,0.025624,0.307483,0.182669,1.683278,8.583070,5.917585,-0.330912,0.753205,312
6,MLP_40,e6a25033ee712d15,0.002287,1356639,post_model_v6_fractional_ties,0.002552,0.060230,0.064485,0.089489,2.331493,...,0.990752,0.025333,0.304001,0.177616,1.711557,8.727261,5.805627,-0.289474,0.721154,312
7,NN3_20,739c8c490c1c6d16,0.001806,1356639,post_model_v6_fractional_ties,0.002120,0.051122,0.054852,0.090022,1.967208,...,0.985011,0.019546,0.234549,0.180191,1.301669,6.637238,4.855011,-0.413451,0.692308,312
8,LASSO_20,bc11f57268346015,0.001746,1356639,post_model_v6_fractional_ties,0.002109,0.071919,0.076510,0.133655,1.864029,...,0.987852,0.014327,0.171923,0.240600,0.714560,3.643554,2.969408,-0.567106,0.467949,312
9,HYBRID_MLP40_DEEPSET40,e76569f5ea5b351a,0.001529,1356639,post_model_v6_fractional_ties,0.001416,0.057928,0.061570,0.083942,2.390546,...,0.992155,0.024872,0.298459,0.170882,1.746579,8.905839,5.839902,-0.354444,0.733974,312


## 7. Portfolio implementability robustness

This cached diagnostic reads each pooled prediction file once and never loads model weights. It evaluates the 10% tail portfolio under full/ex-microcap universes, equal/value weighting, fixed proportional transaction-cost scenarios, and the adverse missing-return stress.

In [ ]:
from src.portfolio_robustness import run_portfolio_robustness

ROBUSTNESS_MODEL_IDS = (
    'LGBM_40',
    'DEEPSET_40_DYNAMIC',
    'HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50',
)

portfolio_robustness = run_portfolio_robustness(
    CONFIG.run_dir,
    model_ids=ROBUSTNESS_MODEL_IDS,
)
display(portfolio_robustness)

LGBM_40: current robustness; skipping
DEEPSET_40_DYNAMIC: saved portfolio robustness
HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50: saved portfolio robustness


,universe,weighting,mean_monthly_turnover,annualized_turnover,mean_n_eligible,mean_long_coverage,mean_short_coverage,gross_mean_monthly_return,gross_annualized_return,gross_annualized_volatility,...,outlier_exclude_abs_gt_10_sharpe,outlier_exclude_abs_gt_10_t_stat,outlier_exclude_abs_gt_10_newey_west_t_stat,outlier_exclude_abs_gt_10_max_drawdown,outlier_exclude_abs_gt_10_hit_rate,outlier_exclude_abs_gt_10_n_months,n_market_abs_gt_10,n_selected_abs_gt_10,model_id,model_signature
4,EX_MICRO,EQUAL,1.183841,14.206098,2284.227564,0.991476,0.994021,0.014417,0.173007,0.180433,...,0.958842,4.889154,3.892569,-0.391687,0.657051,312,1,0,DEEPSET_40_DYNAMIC,4a51c6755ddf965f
5,EX_MICRO,VALUE,1.396020,16.752235,2284.227564,0.991476,0.994021,0.010148,0.121775,0.201062,...,0.605659,3.088265,3.023649,-0.408595,0.592949,312,1,0,DEEPSET_40_DYNAMIC,4a51c6755ddf965f
6,FULL,EQUAL,1.151408,13.816897,4348.201923,0.989532,0.984097,0.026060,0.312722,0.162586,...,1.957261,9.980112,6.373144,-0.328509,0.756410,312,9,2,DEEPSET_40_DYNAMIC,4a51c6755ddf965f
7,FULL,VALUE,1.412021,16.944257,4348.201923,0.989532,0.984097,0.016507,0.198087,0.223291,...,0.891912,4.547878,4.068427,-0.476804,0.628205,312,9,2,DEEPSET_40_DYNAMIC,4a51c6755ddf965f
8,EX_MICRO,EQUAL,1.056904,12.682847,2284.227564,0.991203,0.993288,0.015652,0.187819,0.197509,...,0.950940,4.848862,3.647439,-0.383989,0.647436,312,1,0,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,d14e8c2a63ec5c8e
9,EX_MICRO,VALUE,1.274706,15.296473,2284.227564,0.991203,0.993288,0.012634,0.151609,0.208851,...,0.725918,3.701472,3.164281,-0.487074,0.612179,312,1,0,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,d14e8c2a63ec5c8e
10,FULL,EQUAL,1.031025,12.372303,4348.201923,0.989031,0.979709,0.028445,0.341340,0.168209,...,2.041172,10.407976,6.410485,-0.351370,0.769231,312,9,1,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,d14e8c2a63ec5c8e
11,FULL,VALUE,1.285147,15.421766,4348.201923,0.989031,0.979709,0.017480,0.209758,0.224105,...,0.938501,4.785432,3.721010,-0.542289,0.634615,312,9,1,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,d14e8c2a63ec5c8e
0,EX_MICRO,EQUAL,0.896191,10.754294,2284.227564,0.991412,0.992692,0.012011,0.144137,0.221326,...,0.667078,3.401441,2.681841,-0.421071,0.592949,312,1,1,LGBM_40,9c0540e864a705bf
1,EX_MICRO,VALUE,1.038877,12.466526,2284.227564,0.991412,0.992692,0.012698,0.152372,0.234687,...,0.653900,3.334247,2.811860,-0.512802,0.570513,312,1,1,LGBM_40,9c0540e864a705bf


## 8. Paired model tests

Declared model pairs are evaluated on matched out-of-sample months using differences in monthly MSE, rank IC, 10% tail returns, and Sharpe ratios. Newey--West inference, test-year clustering, year-block bootstrap intervals, Holm adjustment, and a one-standard-error interpretation are reported. Missing artifacts cause the comparison to fail rather than silently omit a planned pair.

In [ ]:
from src.model_comparison import run_paired_model_comparisons

paired_comparison = run_paired_model_comparisons(CONFIG.run_dir, seed=CONFIG.seed)
display(paired_comparison)

,model_a,model_b,difference_definition,mean_monthly_mse_difference,mse_difference_nw_t_stat,mse_difference_p_value,mean_monthly_return_difference,return_difference_nw_t_stat,return_difference_p_value,mean_rank_ic_difference,...,sharpe_difference,sharpe_difference_bootstrap_standard_error,sharpe_difference_exceeds_one_standard_error,sharpe_difference_block_bootstrap_ci_low,sharpe_difference_block_bootstrap_ci_high,simpler_model_within_sharpe_interval,n_paired_months,mse_difference_holm_p_value,return_difference_holm_p_value,rank_ic_difference_holm_p_value
0,NN4_20,NN2_20,model_a_minus_model_b,0.000038,1.407567,0.159259,-0.007963,-3.794501,0.000148,-0.006181,...,-0.594612,0.149661,False,-0.929084,-0.342423,False,312,1.000000,0.002367,1.000000
1,NN4_20,NN3_20,model_a_minus_model_b,0.000085,2.623588,0.008701,-0.010344,-4.050927,0.000051,-0.011555,...,-0.735253,0.172717,False,-1.107387,-0.430349,False,312,0.147915,0.000867,1.000000
2,LGBM_40,LGBM_20,model_a_minus_model_b,0.000012,0.443049,0.657731,0.001076,0.701790,0.482810,-0.006923,...,0.190848,0.131719,True,-0.044976,0.471354,True,312,1.000000,1.000000,0.941792
3,LGBM_40,LGBM_40_LAG1,model_a_minus_model_b,-0.000016,-1.075069,0.282344,-0.001115,-1.113947,0.265302,-0.002065,...,-0.015827,0.080330,False,-0.181889,0.133001,True,312,1.000000,1.000000,1.000000
4,LGBM_40,LGBM_40_LAG2,model_a_minus_model_b,-0.000006,-0.404244,0.686033,-0.000993,-0.897316,0.369550,-0.002451,...,0.011664,0.091675,False,-0.169182,0.190178,True,312,1.000000,1.000000,1.000000
5,LGBM_40_LAG1,LGBM_40_LAG2,model_a_minus_model_b,0.000010,1.271530,0.203540,0.000121,0.212501,0.831716,-0.000386,...,0.027491,0.053888,False,-0.090386,0.120851,True,312,1.000000,1.000000,1.000000
6,MLP_40,DEEPSET_40,model_a_minus_model_b,-0.000055,-0.849090,0.395831,0.001143,0.709348,0.478108,0.002471,...,-0.052084,0.120751,False,-0.304396,0.168938,True,312,1.000000,1.000000,1.000000
7,HYBRID_MLP40_DEEPSET40,MLP_40,model_a_minus_model_b,0.000023,0.406830,0.684133,-0.000472,-0.330058,0.741356,-0.002302,...,0.033953,0.147114,False,-0.215661,0.361016,True,312,1.000000,1.000000,1.000000
8,HYBRID_MLP40_DEEPSET40,DEEPSET_40,model_a_minus_model_b,-0.000032,-0.733758,0.463096,0.000671,0.400552,0.688750,0.000169,...,-0.018131,0.130335,False,-0.282185,0.228717,True,312,1.000000,1.000000,1.000000
9,DEEPSET_40,DEEPSET_40_LAG1,model_a_minus_model_b,-0.000045,-1.328941,0.183868,-0.000237,-0.128876,0.897456,-0.000284,...,0.013791,0.206229,False,-0.375326,0.433075,True,312,1.000000,1.000000,1.000000


## 9. Final completion checks and frozen outputs

In [ ]:
# Read current artifacts without invoking model training.
from src.runner import ExperimentRunner
runner = ExperimentRunner(CONFIG)
final_comparison = runner._cumulative_comparison()
expected_models = set(CONFIG.selected_models)
completed_models = set(final_comparison.loc[
    final_comparison['diagnostics_version'].eq(runner.DIAGNOSTICS_VERSION), 'model_id'
])
missing_current = sorted(expected_models - completed_models)
if missing_current:
    raise RuntimeError(f'Models missing current diagnostics: {missing_current}')
expected_oos_months = 12 * (
    CONFIG.universe.end_year - CONFIG.universe.start_year
    - CONFIG.windows.train_years - CONFIG.windows.validation_years + 1
)
selected_comparison = final_comparison.set_index('model_id').loc[list(expected_models)]
if selected_comparison['n_months'].ne(expected_oos_months).any():
    raise RuntimeError('At least one model does not cover the complete OOS portfolio calendar.')
if (selected_comparison['n_signal_months'] + selected_comparison['n_no_signal_months']).ne(expected_oos_months).any():
    raise RuntimeError('At least one model has an incomplete signal-availability accounting.')
required_columns = [
    'robust_oos_r2', 'mean_monthly_rank_ic', 'n_no_signal_months',
    'mean_monthly_calibration_slope', 'tail_5pct_sharpe',
    'tail_10pct_sharpe', 'tail_20pct_sharpe', 'rank_weighted_sharpe',
    'tail_10pct_missing_return_stress_annualized_return',
]
missing_values = final_comparison.set_index('model_id').loc[list(expected_models), required_columns].isna()
if missing_values.any().any():
    print('Legitimate undefined diagnostics require review:')
    display(missing_values[missing_values.any(axis=1)])
else:
    print('All selected models have current complete diagnostics.')
expected_robustness_rows = 4 * len(ROBUSTNESS_MODEL_IDS)
if len(portfolio_robustness) != expected_robustness_rows:
    raise RuntimeError(
        f'Expected {expected_robustness_rows} robustness rows, got {len(portfolio_robustness)}.'
    )
from src.model_comparison import DEFAULT_MODEL_PAIRS
if len(paired_comparison) != len(DEFAULT_MODEL_PAIRS):
    raise RuntimeError('Paired model comparison is incomplete.')
print('Portfolio robustness and paired model comparisons are complete.')

All selected models have current complete diagnostics.
Portfolio robustness and paired model comparisons are complete.
